# Early-Abort Prediction Across Models: ZymCTRL and RITA-small

## What this closes

`locked-results.md` §3g established early-abort on **ProtGPT2 only**: the layer-30 collapse
classifier keeps **94.9%** of its full-sequence ROC-AUC when it can see only the **first 30
residues** (0.740 vs 0.780), and ~0.72 at 20 residues. That is the project's abstention/early-warning
result and it maps directly onto AI4DD **Track 4**'s named topics ("selective prediction",
"abstention", "decision-aware metrics").

It rests on **one model**. This notebook extends it to every other model where the test is
*structurally possible*.

## Which models are possible — and which are not (this is not a budget choice)

Early-abort needs the same thing any classifier needs: **both classes present in the data.** A pool
that never fails (or always fails) has no label variance, so no classifier can be trained or scored
at any truncation length. Per already-locked results:

| Model | Natural collapse | Early-abort testable? |
|---|---|---|
| ProtGPT2 | 53–56% | ✅ **done** — §3g |
| **ZymCTRL** | ~60–66% (§3d, §1h) | ✅ **yes — best case, well-balanced** |
| **RITA-small** | ~84–90% (§3f, §1k) | ⚠️ **yes, but imbalanced** — only ~10–20% non-collapsed |
| p-IgGen | **0.0%** (§3e, §1i, §41) | ❌ **impossible** — zero failures to learn from |
| Mistral-Prot-134M | ~96–99% (§1j) | ❌ **impossible** — zero successes to learn from |

p-IgGen and Mistral-Prot are excluded for the *same structural reason* §3e already documents — this
is not a new limitation, and it is not something a longer run or a bigger N could fix.

**So the ceiling on this question is 3 models, and this notebook covers the 2 that remain.**

## Design — deliberately identical to §3g so the numbers are comparable

- **Same truncations:** first 10 / 20 / 30 residues vs. full sequence.
- **Same pipeline:** teacher-forced re-encoding → mean-pooled layer activations → StandardScaler →
  PCA-20 → RandomForest, `StratifiedShuffleSplit` 15 repeats, 70/30. Unchanged from `42`/§3g.
- **Each model gets its own separately-trained classifier.** This is *not* a transfer test — §3i
  already established zero-shot transfer fails (AUC 0.409). The question here is whether early-abort
  works *within* each model, the same way §3d/§3f showed full-length prediction does.
- **Read layer per model, taken from each model's own best layer in the locked results:**
  ZymCTRL → **layer 30** (§3d, AUC 0.741); RITA → **layer 6** (§3f, AUC 0.627).
- **N=200 per model** (matching §3d/§3f's own N, not §3g's 250 — these are the pools those sections
  used, so the full-length row here should land near their locked AUCs as a built-in sanity check).

**Ordering is deliberate: ZymCTRL runs and saves completely before RITA is even loaded.** RITA has
the worst reliability history in this project (five separate compatibility issues across `14`–`17`),
so if it fails, ZymCTRL's result is already printed and written to CSV.

## RITA-specific handling (unchanged from `32`/`33`, all five known issues built in)

Scoped monkeypatch with save/restore, forced `float32`, hook path `model.transformer.layers[L]`,
hand-written sampling loop (RITA has no `.generate()`), and **forward-hook activation capture**
(RITA's `output_hidden_states` does not return the standard tuple). Feature extraction therefore
uses hooks for RITA and `output_hidden_states` for ZymCTRL — both mean-pool the same tensor.

Kaggle setup: Accelerator = **GPU T4 x1 or x2**, Internet = **ON**. Expect ~2.5–3.5 hours
(400 folds total + 2 model loads + 8 truncation × feature-extraction passes).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score

torch.manual_seed(2024)
np.random.seed(2024)

device = "cuda" if torch.cuda.is_available() else "cpu"
N_POOL = 200
TRUNC_LENGTHS = [10, 20, 30, None]      # None = full sequence, matching 42/§3g
N_REPEATS = 15                           # matching 42/§3g

# Each model's own best read-layer, from its own locked result -- NOT copied from ProtGPT2.
ZYM_LAYER = 30    # §3d: ZymCTRL best layer 30, AUC 0.741
RITA_LAYER = 6    # §3f: RITA best layer 6 of 12, AUC 0.627

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

print("Setup complete. CUDA available:", torch.cuda.is_available())
print(f"Truncations: {TRUNC_LENGTHS} | N={N_POOL}/model | {N_REPEATS} repeats")
print(f"Read layers -- ZymCTRL: {ZYM_LAYER}, RITA: {RITA_LAYER}")


Setup complete. CUDA available: True
Truncations: [10, 20, 30, None] | N=200/model | 15 repeats
Read layers -- ZymCTRL: 30, RITA: 6


In [2]:
# --- Shared: length-aware ESMFold evaluator (39/40/41/42 convention) and the classifier
#     evaluation function, byte-identical in behaviour to 42's so §3g is directly comparable. ---

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class SafeStructuralEvaluator:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def _fold(self, s):
        inputs = self.tokenizer([s], return_tensors="pt", add_special_tokens=False).to(self.device)
        with torch.no_grad():
            out = self.model(**inputs)
        raw = float(np.mean(out.plddt.cpu().numpy()))
        plddt = raw * 100.0 if raw <= 1.5 else raw
        ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
        return plddt, ptm

    def fold_one(self, seq, max_len=300):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0, False
        working = cleaned[:max_len] if len(cleaned) > max_len else cleaned
        try:
            p, t = self._fold(working)
            return p, t, True
        except RuntimeError:
            clear_gpu()
        half = max(10, len(working) // 2)
        if half < len(working):
            try:
                p, t = self._fold(working[:half])
                return p, t, True
            except RuntimeError:
                clear_gpu()
        return 0.0, 0.0, False

def fold_batch(records, evaluator):
    for r in records:
        p, t, ok = evaluator.fold_one(r["sequence"])
        r["plddt"], r["ptm"], r["fold_ok"] = p, t, ok
        r["collapse"] = int(0.0 < p < 60.0)
    return records

def fit_eval(X, y, n_repeats=N_REPEATS, test_size=0.3, seed=99):
    aucs = []
    sss = StratifiedShuffleSplit(n_splits=n_repeats, test_size=test_size, random_state=seed)
    for tr, te in sss.split(X, y):
        y_tr, y_te = y[tr], y[te]
        if len(set(y_tr)) < 2 or len(set(y_te)) < 2:
            continue
        sc = StandardScaler().fit(X[tr])
        Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
        nc = min(20, Xtr.shape[0] - 1, Xtr.shape[1])
        pca = PCA(n_components=nc, random_state=seed).fit(Xtr)
        clf = RandomForestClassifier(n_estimators=200, random_state=seed)
        clf.fit(pca.transform(Xtr), y_tr)
        aucs.append(roc_auc_score(y_te, clf.predict_proba(pca.transform(Xte))[:, 1]))
    return np.array(aucs)

def report_early_abort(model_name, feats_by_trunc, labels_by_trunc, reference_auc=None):
    rows, auc_by = [], {}
    print(f"\n{'=' * 78}")
    print(f"EARLY-ABORT RESULT -- {model_name}")
    print(f"{'=' * 78}")
    print(f"{'Truncation':>14s} {'N':>5s} {'collapse%':>10s} {'mean AUC':>10s} {'std':>7s}")
    print("-" * 52)
    for t in TRUNC_LENGTHS:
        X = np.array(feats_by_trunc[t]); y = np.array(labels_by_trunc[t])
        if len(y) == 0 or len(set(y)) < 2:
            print(f"{str(t):>14s} -- SKIPPED (no label variance)")
            auc_by[t] = np.array([]); continue
        a = fit_eval(X, y); auc_by[t] = a
        label = f"{t} residues" if t is not None else "full sequence"
        print(f"{label:>14s} {len(y):5d} {y.mean()*100:9.1f}% {a.mean():10.3f} {a.std():7.3f}")
        for v in a:
            rows.append({"model": model_name, "truncation": (t if t is not None else -1), "auc": v})
    full = auc_by.get(None, np.array([]))
    if len(full) and len(auc_by.get(30, [])):
        ret = auc_by[30].mean() / full.mean()
        print(f"\nRetention at 30 residues: {ret:.1%} of full-sequence AUC "
              f"({auc_by[30].mean():.3f} / {full.mean():.3f})")
        print(f"ProtGPT2 reference (§3g): 94.9% retention (0.740 / 0.780)")
        if ret > 0.85:
            print("  ==> Early-abort VIABLE on this model -- most signal is present in the first 30 residues.")
        else:
            print("  ==> Early-abort NOT well supported here -- signal arrives late. Report as a")
            print("      model-specific limitation, not a failure of the method.")
    if reference_auc is not None and len(full):
        print(f"\nSanity check -- full-sequence AUC {full.mean():.3f} vs this model's locked "
              f"full-length value {reference_auc:.3f} (should be close; large gap = investigate)")
    return rows, auc_by

print("Shared evaluator + classifier utilities ready.")


Shared evaluator + classifier utilities ready.


In [3]:
# --- MODEL 1: ZymCTRL. Runs and saves COMPLETELY before RITA is touched. ---
#     EC-number prompts and output cleaning identical to 26/30/38/40.

EC_LABELS = ["1.1.1.1", "1.1.1.2", "2.7.1.1", "2.7.1.2", "3.1.1.1", "3.5.1.4",
             "4.1.1.1", "4.2.1.1", "5.1.3.1", "5.3.1.9", "6.1.1.1", "6.3.2.1"]

def build_ec_prompt_pool(labels, n, seed=11):
    rng = np.random.RandomState(seed)
    return [labels[i % len(labels)] for i in rng.permutation(n)]

def clean_zymctrl_output(txt):
    s = txt.split("<sep>", 1)[1] if "<sep>" in txt else txt
    for tok in ["<start>", "<end>", "<|endoftext|>", "<pad>", " "]:
        s = s.replace(tok, "")
    return s

print(f"Loading ZymCTRL on {device}...")
zym_tok = AutoTokenizer.from_pretrained("AI4PD/ZymCTRL")
zym_model = AutoModelForCausalLM.from_pretrained("AI4PD/ZymCTRL").to(device)
zym_model.eval()
ZEOS = zym_tok.eos_token_id if zym_tok.eos_token_id is not None else 1
ZPAD = zym_tok.pad_token_id if zym_tok.pad_token_id is not None else ZEOS

torch.manual_seed(4040)
zym_records = []
print(f"=== Generating N={N_POOL} natural ZymCTRL sequences ===")
for prompt in build_ec_prompt_pool(EC_LABELS, N_POOL, seed=4040):
    inputs = zym_tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out_ids = zym_model.generate(**inputs, max_length=50, do_sample=True,
                                     temperature=1.2, eos_token_id=ZEOS, pad_token_id=ZPAD)
    seq = clean_zymctrl_output(zym_tok.decode(out_ids[0], skip_special_tokens=False))
    zym_records.append({"prompt": prompt, "sequence": seq, "gen_only": seq})
clear_gpu()

print("=== Freeing ZymCTRL while ESMFold folds ===")
del zym_model
clear_gpu()

ev = SafeStructuralEvaluator()
print("Folding ZymCTRL pool...")
zym_records = fold_batch(zym_records, ev)
del ev
clear_gpu()

zym_valid = [r for r in zym_records if r["fold_ok"]]
print(f"\n{len(zym_valid)}/{len(zym_records)} folded successfully. "
      f"Natural collapse: {np.mean([r['collapse'] for r in zym_valid]):.1%}")
print("(§3d/§1h recorded ~60-66% -- a close value confirms this pool is comparable.)")


Loading ZymCTRL on cuda...


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.88G [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding ZymCTRL pool...

200/200 folded successfully. Natural collapse: 62.5%
(§3d/§1h recorded ~60-66% -- a close value confirms this pool is comparable.)


In [4]:
# --- ZymCTRL feature extraction at each truncation. Standard output_hidden_states path
#     (works fine for GPT-2-family models). Truncation applies to the generated sequence. ---

print(f"Reloading ZymCTRL to extract layer-{ZYM_LAYER} features...")
zym_model = AutoModelForCausalLM.from_pretrained("AI4PD/ZymCTRL").to(device)
zym_model.eval()

def zym_feature(seq, trunc):
    s = seq if trunc is None else seq[:trunc]
    if len(s) < 5:
        return None
    inputs = zym_tok(s, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = zym_model(**inputs, output_hidden_states=True)
    return out.hidden_states[ZYM_LAYER].mean(dim=1).squeeze(0).cpu().numpy()

zym_feats = {t: [] for t in TRUNC_LENGTHS}
zym_labels = {t: [] for t in TRUNC_LENGTHS}
for t in TRUNC_LENGTHS:
    for r in zym_valid:
        f = zym_feature(r["sequence"], t)
        if f is not None:
            zym_feats[t].append(f); zym_labels[t].append(r["collapse"])
    print(f"  trunc={t}: {len(zym_feats[t])} vectors")

del zym_model
clear_gpu()

zym_rows, zym_auc = report_early_abort("ZymCTRL", zym_feats, zym_labels, reference_auc=0.741)

# Save ZymCTRL immediately -- before RITA is loaded, so a RITA failure cannot lose this.
pd.DataFrame(zym_rows).to_csv("early_abort_multimodel_zymctrl.csv", index=False)
pd.DataFrame([{
    "model": "ZymCTRL", "n": len(zym_valid),
    "collapse_rate": float(np.mean([r["collapse"] for r in zym_valid])),
    "read_layer": ZYM_LAYER,
    **{f"auc_{('full' if t is None else t)}": (zym_auc[t].mean() if len(zym_auc[t]) else float("nan"))
       for t in TRUNC_LENGTHS},
}]).to_csv("early_abort_multimodel_zymctrl_summary.csv", index=False)
print("\n[SAVED] early_abort_multimodel_zymctrl{,_summary}.csv -- ZymCTRL is banked.")


Reloading ZymCTRL to extract layer-30 features...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: AI4PD/ZymCTRL
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  trunc=10: 200 vectors
  trunc=20: 200 vectors
  trunc=30: 200 vectors
  trunc=None: 200 vectors

EARLY-ABORT RESULT -- ZymCTRL
    Truncation     N  collapse%   mean AUC     std
----------------------------------------------------
   10 residues   200      62.5%      0.597   0.070
   20 residues   200      62.5%      0.643   0.040
   30 residues   200      62.5%      0.646   0.036
 full sequence   200      62.5%      0.658   0.041

Retention at 30 residues: 98.1% of full-sequence AUC (0.646 / 0.658)
ProtGPT2 reference (§3g): 94.9% retention (0.740 / 0.780)
  ==> Early-abort VIABLE on this model -- most signal is present in the first 30 residues.

Sanity check -- full-sequence AUC 0.658 vs this model's locked full-length value 0.741 (should be close; large gap = investigate)

[SAVED] early_abort_multimodel_zymctrl{,_summary}.csv -- ZymCTRL is banked.


In [5]:
# --- MODEL 2: RITA-small. All five known compatibility issues handled, unchanged from 32/33. ---

def load_rita(device):
    import transformers.modeling_utils as _mu
    _had_m = "mark_tied_weights_as_initialized" in _mu.PreTrainedModel.__dict__
    _orig_m = _mu.PreTrainedModel.__dict__.get("mark_tied_weights_as_initialized")
    _had_a = "all_tied_weights_keys" in _mu.PreTrainedModel.__dict__
    _orig_a = _mu.PreTrainedModel.__dict__.get("all_tied_weights_keys")
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = lambda self: None
    _mu.PreTrainedModel.all_tied_weights_keys = property(lambda self: {})
    try:
        tok = AutoTokenizer.from_pretrained("lightonai/RITA_s")
        m = AutoModelForCausalLM.from_pretrained(
            "lightonai/RITA_s", trust_remote_code=True, torch_dtype=torch.float32).to(device)
        m.eval()
    finally:
        if _had_m: _mu.PreTrainedModel.mark_tied_weights_as_initialized = _orig_m
        else: del _mu.PreTrainedModel.mark_tied_weights_as_initialized
        if _had_a: _mu.PreTrainedModel.all_tied_weights_keys = _orig_a
        else: del _mu.PreTrainedModel.all_tied_weights_keys
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token = tok.eos_token
    return tok, m

def manual_generate(model, input_ids, max_len, temperature=1.2):
    gen = input_ids
    with torch.no_grad():
        for _ in range(max_len - input_ids.shape[1]):
            logits = model(gen).logits[:, -1, :] / temperature
            nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            gen = torch.cat([gen, nxt], dim=1)
    return gen

UNIPROT = ["P0CG48","P00720","P02144","P42212","P01308","P61823",
           "P00648","P99999","P69905","P68871","P00698","P00441"]
FALLBACK = ["NLYIQWLKDGGPSSGRPPPS","LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF","GSQIGAKNTGQVQLNLLAL",
            "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
            "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
            "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV"]

print("Fetching UniProt prefixes for RITA...")
refs = []
for acc in UNIPROT:
    try:
        with urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta", timeout=10) as rsp:
            lines = [l for l in rsp.read().decode("utf-8").strip().split("\n") if l]
        s = "".join(lines[1:])
        if len(s) >= 20:
            refs.append((acc, s))
    except Exception as e:
        print(f"  skip {acc}: {e}")
if not refs:
    print("!! UniProt fetch failed -- Internet toggle likely OFF. Using hardcoded fallback.")
    refs = [(f"local{i+1}", s) for i, s in enumerate(FALLBACK)]

rng = np.random.RandomState(5050)
rita_prefixes = []
for i in range(N_POOL):
    acc, s = refs[i % len(refs)]
    pl = rng.randint(10, 16)
    st = rng.randint(0, max(1, len(s) - pl))
    rita_prefixes.append(s[st:st + pl])

print(f"Loading RITA-small on {device}...")
rita_tok, rita_model = load_rita(device)
N_RITA_LAYERS = len(rita_model.transformer.layers)
print(f"RITA has {N_RITA_LAYERS} layers; reading layer {RITA_LAYER} (§3f's best).")

torch.manual_seed(5050)
rita_records = []
print(f"=== Generating N={N_POOL} natural RITA sequences (manual loop -- slower) ===")
for prompt in rita_prefixes:
    ids = rita_tok(prompt, return_tensors="pt").input_ids.to(device)
    out = manual_generate(rita_model, ids, max_len=50)
    seq = rita_tok.decode(out[0], skip_special_tokens=True).replace(" ", "")
    gen_only = seq[len(prompt):] if seq.startswith(prompt) else seq
    rita_records.append({"prompt": prompt, "sequence": seq, "gen_only": gen_only})
clear_gpu()

print("=== Freeing RITA while ESMFold folds ===")
del rita_model
clear_gpu()

ev2 = SafeStructuralEvaluator()
print("Folding RITA pool...")
rita_records = fold_batch(rita_records, ev2)
del ev2
clear_gpu()

rita_valid = [r for r in rita_records if r["fold_ok"]]
rita_cr = np.mean([r["collapse"] for r in rita_valid]) if rita_valid else float("nan")
n_healthy = sum(1 for r in rita_valid if not r["collapse"])
print(f"\n{len(rita_valid)}/{len(rita_records)} folded. Natural collapse: {rita_cr:.1%} "
      f"({n_healthy} non-collapsed)")
print("(§3f recorded 89.5% -- expect severe imbalance; that is the known limitation, not a bug.)")
if n_healthy < 10:
    print("!! WARNING: fewer than 10 non-collapsed examples. AUCs below will be very high-variance;")
    print("!! report RITA as underpowered rather than as a clean positive or negative.")


Fetching UniProt prefixes for RITA...
Loading RITA-small on cuda...


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

The repository lightonai/RITA_s contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/lightonai/RITA_s .
 You can inspect the repository content at https://hf.co/lightonai/RITA_s.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


rita_configuration.py:   0%|          | 0.00/861 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/lightonai/RITA_s:
- rita_configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

RITA has 12 layers; reading layer 6 (§3f's best).
=== Generating N=200 natural RITA sequences (manual loop -- slower) ===
=== Freeing RITA while ESMFold folds ===
Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding RITA pool...

200/200 folded. Natural collapse: 90.5% (19 non-collapsed)
(§3f recorded 89.5% -- expect severe imbalance; that is the known limitation, not a bug.)


In [6]:
# --- RITA feature extraction: forward hooks (output_hidden_states does NOT return the standard
#     tuple on this model -- issue #5 from the 14-17 debugging rounds). ---

print(f"Reloading RITA to extract layer-{RITA_LAYER} features via forward hook...")
rita_tok, rita_model = load_rita(device)
_cap = {}

def _hook(module, inp, out):
    h = out[0] if isinstance(out, (tuple, list)) else out
    _cap["h"] = h.detach()

handle = rita_model.transformer.layers[RITA_LAYER].register_forward_hook(_hook)

def rita_feature(prompt, gen_only, trunc):
    g = gen_only if trunc is None else gen_only[:trunc]
    full = prompt + g
    if len(full) < 5:
        return None
    ids = rita_tok(full, return_tensors="pt", truncation=True, max_length=256).input_ids.to(device)
    _cap.clear()
    with torch.no_grad():
        rita_model(ids)
    if "h" not in _cap:
        return None
    return _cap["h"].float().mean(dim=1).squeeze(0).cpu().numpy()

rita_feats = {t: [] for t in TRUNC_LENGTHS}
rita_labels = {t: [] for t in TRUNC_LENGTHS}
try:
    for t in TRUNC_LENGTHS:
        for r in rita_valid:
            f = rita_feature(r["prompt"], r["gen_only"], t)
            if f is not None:
                rita_feats[t].append(f); rita_labels[t].append(r["collapse"])
        print(f"  trunc={t}: {len(rita_feats[t])} vectors")
finally:
    handle.remove()

del rita_model
clear_gpu()

rita_rows, rita_auc = report_early_abort("RITA-small", rita_feats, rita_labels, reference_auc=0.627)

pd.DataFrame(rita_rows).to_csv("early_abort_multimodel_rita.csv", index=False)
pd.DataFrame([{
    "model": "RITA-small", "n": len(rita_valid), "collapse_rate": float(rita_cr),
    "n_healthy": int(n_healthy), "read_layer": RITA_LAYER,
    **{f"auc_{('full' if t is None else t)}": (rita_auc[t].mean() if len(rita_auc[t]) else float("nan"))
       for t in TRUNC_LENGTHS},
}]).to_csv("early_abort_multimodel_rita_summary.csv", index=False)
print("\n[SAVED] early_abort_multimodel_rita{,_summary}.csv")


Reloading RITA to extract layer-6 features via forward hook...


The repository lightonai/RITA_s contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/lightonai/RITA_s .
 You can inspect the repository content at https://hf.co/lightonai/RITA_s.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

  trunc=10: 200 vectors
  trunc=20: 200 vectors
  trunc=30: 200 vectors
  trunc=None: 200 vectors

EARLY-ABORT RESULT -- RITA-small
    Truncation     N  collapse%   mean AUC     std
----------------------------------------------------
   10 residues   200      90.5%      0.503   0.142
   20 residues   200      90.5%      0.496   0.142
   30 residues   200      90.5%      0.495   0.107
 full sequence   200      90.5%      0.506   0.109

Retention at 30 residues: 97.7% of full-sequence AUC (0.495 / 0.506)
ProtGPT2 reference (§3g): 94.9% retention (0.740 / 0.780)
  ==> Early-abort VIABLE on this model -- most signal is present in the first 30 residues.

Sanity check -- full-sequence AUC 0.506 vs this model's locked full-length value 0.627 (should be close; large gap = investigate)

[SAVED] early_abort_multimodel_rita{,_summary}.csv


In [7]:
# --- Combined verdict across all three models. ---

PROTGPT2_REF = {10: 0.682, 20: 0.720, 30: 0.740, None: 0.780}   # §3g, locked

print("=" * 88)
print("EARLY-ABORT ACROSS MODELS -- mean ROC-AUC by residues visible")
print("=" * 88)
print(f"{'Model':>14s} {'10 res':>9s} {'20 res':>9s} {'30 res':>9s} {'full':>9s} {'30/full':>9s}")
print("-" * 66)

def _row(name, d, is_ref=False):
    def g(t):
        if is_ref: return d[t]
        a = d.get(t, np.array([]))
        return a.mean() if len(a) else float("nan")
    v10, v20, v30, vf = g(10), g(20), g(30), g(None)
    ret = (v30 / vf) if (vf and not np.isnan(vf) and not np.isnan(v30)) else float("nan")
    print(f"{name:>14s} {v10:9.3f} {v20:9.3f} {v30:9.3f} {vf:9.3f} {ret:8.1%}")
    return {"model": name, "auc_10": v10, "auc_20": v20, "auc_30": v30,
            "auc_full": vf, "retention_30": ret}

summary_rows = [
    _row("ProtGPT2 (§3g)", PROTGPT2_REF, is_ref=True),
    _row("ZymCTRL", zym_auc),
    _row("RITA-small", rita_auc),
]
pd.DataFrame(summary_rows).to_csv("early_abort_multimodel_combined.csv", index=False)

print()
print("Structurally impossible (documented, not skipped for budget):")
print("  p-IgGen        -- 0.0% natural collapse (§3e/§1i/NB41): no failures to learn from")
print("  Mistral-Prot   -- ~96-99% natural collapse (§1j): no successes to learn from")

print()
print("=" * 88)
print("VERDICT")
print("=" * 88)
viable = [r for r in summary_rows if not np.isnan(r["retention_30"]) and r["retention_30"] > 0.85]
print(f"Models where early-abort retains >85% of full-sequence AUC at 30 residues: "
      f"{len(viable)}/{len(summary_rows)}")
for r in summary_rows:
    if np.isnan(r["retention_30"]):
        print(f"  {r['model']}: NOT EVALUABLE")
    else:
        print(f"  {r['model']}: {r['retention_30']:.1%} retention "
              f"({'viable' if r['retention_30'] > 0.85 else 'signal arrives late'})")
print()
if len(viable) == 3:
    print("  ==> Early-abort generalizes across all three testable models. This upgrades §3g from a")
    print("      single-model result to a 3-model one, and it is the strongest possible version of")
    print("      the abstention claim given p-IgGen and Mistral-Prot are structurally untestable.")
    print("      Report the ceiling honestly: 3 of 5 models CAN be tested, and all 3 work.")
elif len(viable) >= 2:
    print("  ==> Early-abort holds on some models but not all. Report per-model rather than as a")
    print("      general property -- consistent with §3i's finding that the collapse signal is")
    print("      model-specific. Still a meaningful upgrade over the single-model §3g.")
else:
    print("  ==> Early-abort does NOT generalize beyond ProtGPT2. This is a real negative and it")
    print("      must scope §3g's claim to ProtGPT2 explicitly in the paper. Combined with §3i")
    print("      (transfer fails), the honest story becomes: the collapse signal is strongly")
    print("      model-specific in both WHERE it lives and WHEN it becomes readable.")
print()
print("Note for RITA specifically: check its non-collapsed count above. With ~10-20% healthy")
print("examples, wide AUC spreads are expected and a null there is underpowered, not disproof.")


EARLY-ABORT ACROSS MODELS -- mean ROC-AUC by residues visible
         Model    10 res    20 res    30 res      full   30/full
------------------------------------------------------------------
ProtGPT2 (§3g)     0.682     0.720     0.740     0.780    94.9%
       ZymCTRL     0.597     0.643     0.646     0.658    98.1%
    RITA-small     0.503     0.496     0.495     0.506    97.7%

Structurally impossible (documented, not skipped for budget):
  p-IgGen        -- 0.0% natural collapse (§3e/§1i/NB41): no failures to learn from
  Mistral-Prot   -- ~96-99% natural collapse (§1j): no successes to learn from

VERDICT
Models where early-abort retains >85% of full-sequence AUC at 30 residues: 3/3
  ProtGPT2 (§3g): 94.9% retention (viable)
  ZymCTRL: 98.1% retention (viable)
  RITA-small: 97.7% retention (viable)

  ==> Early-abort generalizes across all three testable models. This upgrades §3g from a
      single-model result to a 3-model one, and it is the strongest possible version of
    